# TP2 – Analyse de ventes e-commerce avec Spark SQL : CORRIGÉ
*Nettoyage, typage, jointures, agrégations, fonctions de fenêtre et optimisation Catalyst*




## Objectifs
- Charger des fichiers CSV et JSON avec un schéma explicite et auditer la qualité des données.
- Corriger des données sales : doublons, valeurs manquantes, types invalides, casse et espaces.
- Typer correctement les colonnes en tenant compte du mode ANSI de Spark 4 (try_cast).
- Réaliser des jointures et des agrégations en API DataFrame et en SQL.
- Utiliser les fonctions de fenêtre (classement, cumul).
- Lire un plan d'exécution (explain), exploiter cache, broadcast et partitionnement Parquet.

## Contexte

Vous êtes data analyst pour une boutique en ligne marocaine. Trois sources de données vous sont fournies : le catalogue des produits (CSV), les clients (JSON) et les commandes de l'année 2025 (CSV). Ces données proviennent de systèmes différents et contiennent volontairement des erreurs que vous devrez détecter et corriger avant toute analyse. Objectif final : produire un rapport de chiffre d'affaires (CA) fiable.

## Partie 0 – Génération des données

Exécutez la cellule suivante pour créer le dossier data_tp2 avec les trois fichiers (produits.csv, clients.json, commandes.csv). Le jeu de données est reproductible (graine aléatoire fixe) : tous les étudiants obtiennent les mêmes résultats.

In [1]:
import os, csv, json, random
from datetime import date, timedelta

random.seed(42)
os.makedirs("data_tp2", exist_ok=True)

# ---------- produits.csv ----------
catalogue = {
    "Informatique": ["Clavier", "Souris", "Ecran", "Casque", "Webcam"],
    "Maison":       ["Lampe", "Chaise", "Bureau", "Etagere"],
    "Sport":        ["Ballon", "Raquette", "Tapis de yoga", "Haltères"],
}
produits_l, pid = [], 1
for cat, noms in catalogue.items():
    for n in noms:
        produits_l.append([pid, n, cat, random.choice([49, 79, 99, 149, 249, 399, 599])])
        pid += 1
produits_l[3][3] = "N/A"          # prix invalide (produit 4)
with open("data_tp2/produits.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id_produit", "nom", "categorie", "prix"])
    w.writerows(produits_l)

# ---------- clients.json (un objet JSON par ligne) ----------
villes = ["Casablanca", "Rabat", "Fès", "Marrakech", "Tanger", "Agadir"]
prenoms = ["Sara", "Youssef", "Imane", "Omar", "Salma",
           "Hamza", "Khadija", "Anas", "Meryem", "Ayoub"]
familles = ["Alaoui", "Benani", "Cherkaoui", "Darif", "El Idrissi", "Fassi", "Tazi"]
with open("data_tp2/clients.json", "w", encoding="utf-8") as f:
    for i in range(1, 61):
        ville = random.choice(villes)
        if i % 7 == 0:
            ville = " " + ville.upper() + " "     # ville mal saisie
        c = {"id_client": i,
             "nom": random.choice(prenoms) + " " + random.choice(familles),
             "ville": ville,
             "date_inscription": (date(2024, 1, 1)
                                  + timedelta(days=random.randint(0, 364))).isoformat()}
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

# ---------- commandes.csv ----------
statuts = ["livree", "livree", "livree", "annulee", "en_cours", "Livree", "LIVREE "]
cmd = []
for i in range(1, 3001):
    d = date(2025, 1, 1) + timedelta(days=random.randint(0, 364))
    cmd.append([i, random.randint(1, 54), random.randint(1, 13),
                random.randint(1, 5), d.isoformat(), random.choice(statuts)])
for k in range(0, 3000, 150):  cmd[k][3] = "abc"    # quantité invalide
for k in range(5, 3000, 200):  cmd[k][3] = -2       # quantité négative
for k in range(10, 3000, 300): cmd[k][4] = ""       # date manquante
for k in range(20, 3000, 500): cmd[k][1] = ""       # client manquant
cmd += random.sample(cmd, 40)                       # doublons exacts
with open("data_tp2/commandes.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id_commande", "id_client", "id_produit",
                "quantite", "date_commande", "statut"])
    w.writerows(cmd)

print(sorted(os.listdir("data_tp2")))

['clients.json', 'commandes.csv', 'produits.csv']


## Partie 1 – Chargement avec schéma explicite

Les données sales ne doivent pas être typées à l'aveugle : un « abc » dans une colonne numérique ferait échouer une lecture trop stricte. On charge donc tout en chaînes de caractères, on audite, puis on convertit.

### Question 1.1

Créez une SparkSession (appName « TP2 - Ventes », master local[*], spark.sql.shuffle.partitions = 8). Importez functions (alias F) et Window.

*Indice : from pyspark.sql import functions as F ; from pyspark.sql.window import Window*

In [2]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType

spark = (SparkSession.builder
         .appName("TP2 - Ventes")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())

### Question 1.2

Chargez les trois sources dans les DataFrames produits_raw, clients_raw et commandes_raw. Pour les CSV, utilisez un schéma explicite où toutes les colonnes sont de type StringType et l'option header. Affichez schéma, 5 premières lignes et nombre de lignes de chaque DataFrame.

*Indice : Vous pouvez fournir le schéma sous forme de chaîne DDL : .schema("id_produit STRING, nom STRING, ..."). Le JSON est lu avec spark.read.json.*
pas d'inference

In [3]:
produits_raw = (spark.read.option("header", True)
    .schema("id_produit STRING, nom STRING, categorie STRING, prix STRING")
    .csv("data_tp2/produits.csv"))

commandes_raw = (spark.read.option("header", True)
    .schema("id_commande STRING, id_client STRING, id_produit STRING, "
            "quantite STRING, date_commande STRING, statut STRING")
    .csv("data_tp2/commandes.csv"))

clients_raw = spark.read.json("data_tp2/clients.json")

sources = [("produits", produits_raw), ("clients", clients_raw), ("commandes", commandes_raw)]
for nom, d in sources:
    print("=====", nom, "-", d.count(), "lignes")
    d.printSchema()
    d.show(5, truncate=False)

===== produits - 13 lignes
root
 |-- id_produit: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- categorie: string (nullable = true)
 |-- prix: string (nullable = true)

+----------+-------+------------+----+
|id_produit|nom    |categorie   |prix|
+----------+-------+------------+----+
|1         |Clavier|Informatique|399 |
|2         |Souris |Informatique|49  |
|3         |Ecran  |Informatique|49  |
|4         |Casque |Informatique|N/A |
|5         |Webcam |Informatique|99  |
+----------+-------+------------+----+
only showing top 5 rows
===== clients - 60 lignes
root
 |-- date_inscription: string (nullable = true)
 |-- id_client: long (nullable = true)
 |-- nom: string (nullable = true)
 |-- ville: string (nullable = true)

+----------------+---------+---------------+----------+
|date_inscription|id_client|nom            |ville     |
+----------------+---------+---------------+----------+
|2024-01-17      |1        |Ayoub Darif    |Casablanca|
|2024-04-29      |2    

## Partie 2 – Audit de la qualité des données

Avant de corriger, on mesure. Un audit systématique évite de découvrir les problèmes en fin d'analyse.

### Question 2.1

Pour commandes_raw, calculez le nombre de valeurs manquantes (null ou chaîne vide après trim) pour chaque colonne, en une seule requête select.

*Indice : F.count(F.when(condition, 1)) ne compte que les lignes où la condition est vraie. Utilisez une liste en compréhension sur df.columns.*

In [4]:
def audit_nulls(df):
    return df.select([
        F.count(F.when(F.col(c).isNull()
                       | (F.trim(F.col(c).cast("string")) == ""), 1)).alias(c)
        for c in df.columns])

audit_nulls(commandes_raw).show()

+-----------+---------+----------+--------+-------------+------+
|id_commande|id_client|id_produit|quantite|date_commande|statut|
+-----------+---------+----------+--------+-------------+------+
|          0|        7|         0|       0|           10|     0|
+-----------+---------+----------+--------+-------------+------+



### Question 2.2

Combien de commandes sont des doublons exacts ? Combien de valeurs distinctes de statut observe-t-on (avec leurs effectifs) ? Que remarquez-vous ?

*Indice : count() - dropDuplicates().count() ; groupBy("statut").count()*

In [5]:
nb_doublons = commandes_raw.count() - commandes_raw.dropDuplicates().count()
print("Doublons exacts :", nb_doublons)

commandes_raw.groupBy("statut").count().orderBy("statut").show()

Doublons exacts : 40
+--------+-----+
|  statut|count|
+--------+-----+
| LIVREE |  426|
|  Livree|  465|
| annulee|  476|
|en_cours|  413|
|  livree| 1260|
+--------+-----+



**Réponse :** On trouve 40 doublons exacts. Le statut « livree » apparaît sous trois formes (livree, Livree, « LIVREE » avec espace) : il faudra normaliser avec lower() et trim().

### Question 2.3

Détectez les quantités non numériques ou négatives dans commandes_raw. Combien de lignes sont concernées ? Affichez 5 exemples. Utilisez try_cast pour la conversion.

*Indice : F.col("quantite").try_cast("int") retourne null au lieu de lever une erreur en mode ANSI (Spark 4).*

In [6]:
q = commandes_raw.withColumn("q_int", F.col("quantite").try_cast("int"))
invalides = q.filter(F.col("q_int").isNull() | (F.col("q_int") <= 0))
print("Quantités invalides :", invalides.count())
invalides.show(5)

Quantités invalides : 35
+-----------+---------+----------+--------+-------------+-------+-----+
|id_commande|id_client|id_produit|quantite|date_commande| statut|q_int|
+-----------+---------+----------+--------+-------------+-------+-----+
|          1|       51|         8|     abc|   2025-05-05| livree| NULL|
|          6|       35|         8|      -2|   2025-04-08|annulee|   -2|
|        151|       25|         8|     abc|   2025-09-25| livree| NULL|
|        206|       44|         5|      -2|   2025-11-20| Livree|   -2|
|        301|       43|         5|     abc|   2025-09-27| livree| NULL|
+-----------+---------+----------+--------+-------------+-------+-----+
only showing top 5 rows


**Réponse :** On obtient 35 lignes avec une quantité invalide ou négative (valeurs « abc » et -2), doublons compris. Remarque : un cast("int") classique lèverait ici une exception à cause du mode ANSI.

### Question 2.4

Quel est le problème de la colonne prix dans produits_raw ? Identifiez le produit concerné.

In [7]:
produits_raw.withColumn("prix_num", F.col("prix").try_cast("double")) \
            .filter(F.col("prix_num").isNull()).show()

+----------+------+------------+----+--------+
|id_produit|   nom|   categorie|prix|prix_num|
+----------+------+------------+----+--------+
|         4|Casque|Informatique| N/A|    NULL|
+----------+------+------------+----+--------+



**Réponse :** Le produit 4 (Casque) a le prix « N/A » : valeur non numérique à remplacer.

## Partie 3 – Nettoyage et typage

Appliquez les règles de nettoyage suivantes, dans cet ordre, et produisez trois DataFrames propres : produits, clients et commandes. Règles : (R1) supprimer les doublons exacts ; (R2) convertir id_*, quantite et prix en nombres, date_commande et date_inscription en dates ; (R3) normaliser statut et ville (trim, lower pour statut, première lettre en majuscule pour ville) ; (R4) supprimer les commandes dont la quantité est invalide ou ≤ 0, ou dont la date ou le client est manquant ; (R5) remplacer un prix invalide par le prix moyen des produits de la même catégorie.

### Question 3.1

Construisez le DataFrame commandes en appliquant R1 à R4. Affichez le nombre de lignes avant et après nettoyage, ainsi que le schéma final.

*Indice : try_cast("int"), try_cast("date"), F.lower(F.trim(...)). Faites le filtre R4 après le typage.*

In [8]:
commandes = (commandes_raw
    .dropDuplicates()                                              # R1
    .withColumn("id_commande", F.col("id_commande").try_cast("int"))   # R2
    .withColumn("id_client",   F.col("id_client").try_cast("int"))
    .withColumn("id_produit",  F.col("id_produit").try_cast("int"))
    .withColumn("quantite",    F.col("quantite").try_cast("int"))
    .withColumn("date_commande", F.col("date_commande").try_cast("date"))
    .withColumn("statut", F.lower(F.trim(F.col("statut"))))        # R3
    .filter(F.col("quantite").isNotNull() & (F.col("quantite") > 0)  # R4
            & F.col("date_commande").isNotNull()
            & F.col("id_client").isNotNull()))

print("Avant :", commandes_raw.count(), "- Après :", commandes.count())
commandes.printSchema()
commandes.groupBy("statut").count().show()

Avant : 3040 - Après : 2949
root
 |-- id_commande: integer (nullable = true)
 |-- id_client: integer (nullable = true)
 |-- id_produit: integer (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- date_commande: date (nullable = true)
 |-- statut: string (nullable = true)

+--------+-----+
|  statut|count|
+--------+-----+
|en_cours|  397|
| annulee|  463|
|  livree| 2089|
+--------+-----+



### Question 3.2

Construisez le DataFrame produits (R2 et R5) : convertissez id_produit et prix, puis remplacez le prix manquant par la moyenne de sa catégorie à l'aide d'une fonction de fenêtre (Window.partitionBy("categorie")).

*Indice : F.coalesce(F.col("prix"), F.avg("prix").over(w)). La moyenne ignore les null.*

In [9]:
w_cat = Window.partitionBy("categorie")
produits = (produits_raw
    .withColumn("id_produit", F.col("id_produit").try_cast("int"))
    .withColumn("prix", F.col("prix").try_cast("double"))
    .withColumn("prix", F.round(F.coalesce(F.col("prix"), F.avg("prix").over(w_cat)), 2)))
produits.orderBy("id_produit").show()

+----------+-------------+------------+-----+
|id_produit|          nom|   categorie| prix|
+----------+-------------+------------+-----+
|         1|      Clavier|Informatique|399.0|
|         2|       Souris|Informatique| 49.0|
|         3|        Ecran|Informatique| 49.0|
|         4|       Casque|Informatique|149.0|
|         5|       Webcam|Informatique| 99.0|
|         6|        Lampe|      Maison| 79.0|
|         7|       Chaise|      Maison| 79.0|
|         8|       Bureau|      Maison| 79.0|
|         9|      Etagere|      Maison|399.0|
|        10|       Ballon|       Sport| 49.0|
|        11|     Raquette|       Sport|399.0|
|        12|Tapis de yoga|       Sport|399.0|
|        13|     Haltères|       Sport|249.0|
+----------+-------------+------------+-----+



### Question 3.3

Construisez le DataFrame clients (R2 et R3) : id_client en entier, date_inscription en date, ville nettoyée (trim puis initcap). Vérifiez que la liste des villes distinctes est maintenant cohérente (6 valeurs).

*Indice : F.initcap(F.trim(...)). Attention : « FÈS » devient « Fès » avec initcap.*

In [10]:
clients = (clients_raw
    .withColumn("id_client", F.col("id_client").try_cast("int"))
    .withColumn("date_inscription", F.col("date_inscription").try_cast("date"))
    .withColumn("ville", F.initcap(F.trim(F.col("ville")))))
clients.select("ville").distinct().orderBy("ville").show()
print(clients.select("ville").distinct().count())

+----------+
|     ville|
+----------+
|    Agadir|
|Casablanca|
|       Fès|
| Marrakech|
|     Rabat|
|    Tanger|
+----------+

6


## Partie 4 – Requêtes SQL sur vues temporaires

Enregistrez les trois DataFrames propres comme vues temporaires nommées produits, clients et commandes. Dans toute la suite, le chiffre d'affaires (CA) ne compte que les commandes de statut « livree » et vaut quantite × prix.

### Question 4.1

Créez les trois vues temporaires. Écrivez ensuite une requête SQL donnant le CA total des commandes livrées, ainsi que le nombre de commandes livrées.

*Indice : SELECT SUM(c.quantite * p.prix) ... FROM commandes c JOIN produits p ON ...*

In [11]:
produits.createOrReplaceTempView("produits")
clients.createOrReplaceTempView("clients")
commandes.createOrReplaceTempView("commandes")

spark.sql("""
    SELECT COUNT(*) AS nb_commandes_livrees,
           ROUND(SUM(c.quantite * p.prix), 2) AS ca_total
    FROM commandes c
    JOIN produits p ON c.id_produit = p.id_produit
    WHERE c.statut = 'livree'
""").show()

+--------------------+---------+
|nb_commandes_livrees| ca_total|
+--------------------+---------+
|                2089|1215123.0|
+--------------------+---------+



### Question 4.2

Donnez le CA par catégorie de produit, trié par CA décroissant, avec le nombre de commandes et le panier moyen (CA / nombre de commandes).

In [12]:
spark.sql("""
    SELECT p.categorie,
           COUNT(*) AS nb_commandes,
           ROUND(SUM(c.quantite * p.prix), 2) AS ca,
           ROUND(SUM(c.quantite * p.prix) / COUNT(*), 2) AS panier_moyen
    FROM commandes c
    JOIN produits p ON c.id_produit = p.id_produit
    WHERE c.statut = 'livree'
    GROUP BY p.categorie
    ORDER BY ca DESC
""").show()

+------------+------------+--------+------------+
|   categorie|nb_commandes|      ca|panier_moyen|
+------------+------------+--------+------------+
|       Sport|         685|574687.0|      838.96|
|Informatique|         800|354311.0|      442.89|
|      Maison|         604|286125.0|      473.72|
+------------+------------+--------+------------+



### Question 4.3

Donnez le CA mensuel (colonne mois au format yyyy-MM), trié chronologiquement. Quel mois a généré le CA le plus élevé ?

*Indice : date_format(c.date_commande, 'yyyy-MM')*

In [13]:
ca_mensuel = spark.sql("""
    SELECT date_format(c.date_commande, 'yyyy-MM') AS mois,
           ROUND(SUM(c.quantite * p.prix), 2) AS ca
    FROM commandes c
    JOIN produits p ON c.id_produit = p.id_produit
    WHERE c.statut = 'livree'
    GROUP BY date_format(c.date_commande, 'yyyy-MM')
    ORDER BY mois
""")
ca_mensuel.show(12)
ca_mensuel.orderBy(F.desc("ca")).show(1)

+-------+--------+
|   mois|      ca|
+-------+--------+
|2025-01|118017.0|
|2025-02| 98790.0|
|2025-03|109451.0|
|2025-04| 81453.0|
|2025-05| 86787.0|
|2025-06|114252.0|
|2025-07|106081.0|
|2025-08|114002.0|
|2025-09| 92077.0|
|2025-10| 90417.0|
|2025-11| 99390.0|
|2025-12|104406.0|
+-------+--------+

+-------+--------+
|   mois|      ca|
+-------+--------+
|2025-01|118017.0|
+-------+--------+
only showing top 1 row


### Question 4.4

Donnez le top 5 des produits par CA (nom, catégorie, quantité totale vendue, CA).

In [14]:
spark.sql("""
    SELECT p.nom, p.categorie,
           SUM(c.quantite) AS qte_vendue,
           ROUND(SUM(c.quantite * p.prix), 2) AS ca
    FROM commandes c
    JOIN produits p ON c.id_produit = p.id_produit
    WHERE c.statut = 'livree'
    GROUP BY p.nom, p.categorie
    ORDER BY ca DESC
    LIMIT 5
""").show()

+-------------+------------+----------+--------+
|          nom|   categorie|qte_vendue|      ca|
+-------------+------------+----------+--------+
|     Raquette|       Sport|       552|220248.0|
|Tapis de yoga|       Sport|       517|206283.0|
|      Clavier|Informatique|       491|195909.0|
|      Etagere|      Maison|       451|179949.0|
|     Haltères|       Sport|       485|120765.0|
+-------------+------------+----------+--------+



## Partie 5 – Jointures en API DataFrame

Refaites maintenant certaines analyses avec l'API DataFrame. Commencez par construire un DataFrame enrichi regroupant commandes, produits et clients.

### Question 5.1

Construisez le DataFrame ventes : jointure interne de commandes avec produits (sur id_produit) puis avec clients (sur id_client), en ne gardant que les commandes livrées, avec les colonnes id_commande, date_commande, nom_client, ville, nom (produit), categorie, quantite, prix et ca = quantite × prix. Attention aux noms de colonnes en double (nom).

*Indice : Renommez avant la jointure : clients.withColumnRenamed("nom", "nom_client") et produits.withColumnRenamed("nom", "produit").*

In [15]:
ventes = (commandes
    .filter(F.col("statut") == "livree")
    .join(produits.withColumnRenamed("nom", "produit"), "id_produit", "inner")
    .join(clients.withColumnRenamed("nom", "nom_client"), "id_client", "inner")
    .withColumn("ca", F.col("quantite") * F.col("prix"))
    .select("id_commande", "date_commande", "nom_client", "ville",
            "produit", "categorie", "quantite", "prix", "ca"))
ventes.show(5)
print(ventes.count())

+-----------+-------------+-----------------+------+-------------+------------+--------+-----+------+
|id_commande|date_commande|       nom_client| ville|      produit|   categorie|quantite| prix|    ca|
+-----------+-------------+-----------------+------+-------------+------------+--------+-----+------+
|          8|   2025-02-08|      Ayoub Darif|Agadir|     Haltères|       Sport|       5|249.0|1245.0|
|         13|   2025-08-05|     Salma Benani|   Fès|Tapis de yoga|       Sport|       5|399.0|1995.0|
|         18|   2025-02-11|       Ayoub Tazi|Agadir|       Souris|Informatique|       5| 49.0| 245.0|
|         35|   2025-05-09|     Meryem Darif| Rabat|       Souris|Informatique|       4| 49.0| 196.0|
|         82|   2025-09-16|Meryem El Idrissi| Rabat|      Etagere|      Maison|       4|399.0|1596.0|
+-----------+-------------+-----------------+------+-------------+------------+--------+-----+------+
only showing top 5 rows
2089


### Question 5.2

À partir de ventes, calculez le CA par ville (ordre décroissant) en API DataFrame, puis vérifiez que la somme des CA par ville est égale au CA total trouvé en 4.1.

In [16]:
ca_ville = (ventes.groupBy("ville")
            .agg(F.round(F.sum("ca"), 2).alias("ca"), F.count("*").alias("nb_commandes"))
            .orderBy(F.desc("ca")))
ca_ville.show()
print(ca_ville.agg(F.sum("ca")).first()[0])

+----------+--------+------------+
|     ville|      ca|nb_commandes|
+----------+--------+------------+
|     Rabat|253369.0|         412|
|    Tanger|220226.0|         361|
|       Fès|209395.0|         352|
| Marrakech|208387.0|         383|
|    Agadir|170305.0|         290|
|Casablanca|153441.0|         291|
+----------+--------+------------+

1215123.0


### Question 5.3

Quels clients n'ont passé aucune commande (propre) ? Utilisez une jointure de type left_anti. Combien y en a-t-il ?

*Indice : clients.join(commandes, "id_client", "left_anti")*

In [17]:
sans_commande = clients.join(commandes, "id_client", "left_anti")
sans_commande.orderBy("id_client").show()
print(sans_commande.count())

+---------+----------------+------------------+----------+
|id_client|date_inscription|               nom|     ville|
+---------+----------------+------------------+----------+
|       55|      2024-04-22|   Omar El Idrissi|    Tanger|
|       56|      2024-11-19|     Youssef Fassi|Casablanca|
|       57|      2024-01-17|       Omar Alaoui|Casablanca|
|       58|      2024-05-01|Youssef El Idrissi|       Fès|
|       59|      2024-10-03|       Anas Benani|       Fès|
|       60|      2024-08-30|  Ayoub El Idrissi|     Rabat|
+---------+----------------+------------------+----------+

6


**Réponse :** Les clients 55 à 60 n'ont jamais commandé (6 clients). left_anti conserve les lignes de gauche sans correspondance à droite.

### Question 5.4

Expliquez la différence entre inner, left, left_semi et left_anti. Dans quel cas un left join aurait-il donné un résultat différent d'un inner join à la question 5.1 ?

**Réponse :** inner : uniquement les lignes avec correspondance des deux côtés ; left : toutes les lignes de gauche (null à droite si pas de correspondance) ; left_semi : lignes de gauche ayant une correspondance, sans colonnes de droite ; left_anti : lignes de gauche sans correspondance. En 5.1, un left join différerait de l'inner join si une commande référençait un client ou un produit absent des tables de référence (ici, non, car les identifiants existent).

## Partie 6 – Fonctions de fenêtre

Contrairement à un groupBy classique (qui regroupe les lignes et réduit le tableau en une seule ligne par groupe), une fonction de fenêtre calcule une valeur (ou un rang) pour chaque ligne en gardant toutes les lignes d'origine.   

### Question 6.1

Pour chaque catégorie, classez les produits par CA décroissant (dense_rank) et conservez uniquement les 2 meilleurs de chaque catégorie.

F.dense_rank().over(w) : Attribue un rang à chaque produit dans sa catégorie en fonction de son CA. Si deux produits ont exactement le même chiffre d'affaires, ils obtiendront le même rang (contrairement à row_number()).

*Indice : Window.partitionBy("categorie").orderBy(F.desc("ca")) ; F.dense_rank().over(w) ; puis filter(rang <= 2)*
dense_rank() garantit que si deux produits ont exactement le même chiffre d'affaires, ils obtiendront le même rang sans laisser de trous dans la numérotation.

partitionBy("categorie") : On sépare nos données par catégorie (par exemple, un groupe pour l'Informatique, un pour la Maison, etc.). Le classement va se faire séparément dans chaque boîte.

orderBy(F.desc("ca")) : À l'intérieur de chaque catégorie, on trie les produits du plus grand CA au plus petit (décroissant).

In [34]:

ca_produit = ventes.groupBy("categorie", "produit").agg(F.round(F.sum("ca"), 2).alias("ca"))
 
w = Window.partitionBy("categorie").orderBy(F.desc("ca"))
 
(ca_produit.withColumn("rang", F.dense_rank().over(w))
           .filter(F.col("rang") <= 2)
           .orderBy("categorie", "rang")
           .show())

+------------+-------------+--------+----+
|   categorie|      produit|      ca|rang|
+------------+-------------+--------+----+
|Informatique|      Clavier|195909.0|   1|
|Informatique|       Casque| 60941.0|   2|
|      Maison|      Etagere|179949.0|   1|
|      Maison|       Bureau| 39184.0|   2|
|       Sport|     Raquette|220248.0|   1|
|       Sport|Tapis de yoga|206283.0|   2|
+------------+-------------+--------+----+



### Question 6.2

Calculez le CA cumulé mois par mois. Ajoutez aussi la variation en pourcentage par rapport au mois précédent (fonction lag).

*Indice : Window.orderBy("mois") ; F.sum("ca").over(w.rowsBetween(Window.unboundedPreceding, Window.currentRow)) ; F.lag("ca", 1).over(w)*

In [19]:
mensuel = (ventes.withColumn("mois", F.date_format("date_commande", "yyyy-MM"))
                 .groupBy("mois").agg(F.round(F.sum("ca"), 2).alias("ca")))
w = Window.orderBy("mois")
(mensuel
  .withColumn("ca_cumule", F.sum("ca").over(
      w.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
  .withColumn("ca_prec", F.lag("ca", 1).over(w))
  .withColumn("variation_pct",
              F.round((F.col("ca") - F.col("ca_prec")) / F.col("ca_prec") * 100, 1))
  .drop("ca_prec")
  .orderBy("mois")
  .show(12))

+-------+--------+---------+-------------+
|   mois|      ca|ca_cumule|variation_pct|
+-------+--------+---------+-------------+
|2025-01|118017.0| 118017.0|         NULL|
|2025-02| 98790.0| 216807.0|        -16.3|
|2025-03|109451.0| 326258.0|         10.8|
|2025-04| 81453.0| 407711.0|        -25.6|
|2025-05| 86787.0| 494498.0|          6.5|
|2025-06|114252.0| 608750.0|         31.6|
|2025-07|106081.0| 714831.0|         -7.2|
|2025-08|114002.0| 828833.0|          7.5|
|2025-09| 92077.0| 920910.0|        -19.2|
|2025-10| 90417.0|1011327.0|         -1.8|
|2025-11| 99390.0|1110717.0|          9.9|
|2025-12|104406.0|1215123.0|          5.0|
+-------+--------+---------+-------------+



## Partie 7 – Segmentation des clients

L'équipe marketing souhaite segmenter les clients selon leur CA total : « Gold » (CA ≥ 25 000), « Silver » (CA ≥ 20 000) et « Bronze » sinon.

### Question 7.1

Calculez le CA total par client à partir de ventes, puis ajoutez la colonne segment avec when/otherwise. Combien de clients par segment ?

In [20]:
ca_client = (commandes.filter(F.col("statut") == "livree")
    .join(produits, "id_produit")
    .groupBy("id_client")
    .agg(F.round(F.sum(F.col("quantite") * F.col("prix")), 2).alias("ca_total")))

segments = ca_client.withColumn(
    "segment",
    F.when(F.col("ca_total") >= 25000, "Gold")
     .when(F.col("ca_total") >= 20000, "Silver")
     .otherwise("Bronze"))
segments.groupBy("segment").count().orderBy("segment").show()

+-------+-----+
|segment|count|
+-------+-----+
| Bronze|   22|
|   Gold|   16|
| Silver|   16|
+-------+-----+



### Question 7.2

Faites la même segmentation en SQL pur (CASE WHEN) en partant de la vue commandes. Joignez le résultat aux clients pour afficher le nom et la ville des 5 meilleurs clients.

In [21]:
spark.sql("""
    WITH ca AS (
        SELECT c.id_client, SUM(c.quantite * p.prix) AS ca_total
        FROM commandes c JOIN produits p ON c.id_produit = p.id_produit
        WHERE c.statut = 'livree'
        GROUP BY c.id_client
    )
    SELECT cl.nom, cl.ville, ROUND(ca.ca_total, 2) AS ca_total,
           CASE WHEN ca.ca_total >= 25000 THEN 'Gold'
                WHEN ca.ca_total >= 20000 THEN 'Silver'
                ELSE 'Bronze' END AS segment
    FROM ca JOIN clients cl ON ca.id_client = cl.id_client
    ORDER BY ca.ca_total DESC
    LIMIT 5
""").show()

+-----------------+------+--------+-------+
|              nom| ville|ca_total|segment|
+-----------------+------+--------+-------+
|     Hamza Alaoui|   Fès| 41736.0|   Gold|
|Meryem El Idrissi|   Fès| 38512.0|   Gold|
|Khadija Cherkaoui| Rabat| 35376.0|   Gold|
|    Meryem Alaoui|Tanger| 32523.0|   Gold|
| Salma El Idrissi|Agadir| 31485.0|   Gold|
+-----------------+------+--------+-------+



## Partie 8 – Optimisation et plans d'exécution (Catalyst)

Spark SQL s'appuie sur l'optimiseur Catalyst (règles et coûts) et sur le moteur Tungsten. Dans cette partie, vous observez ses décisions grâce à explain().

### Question 8.1

Affichez le plan de la requête suivante avec explain("formatted") : jointure de commandes et produits, puis filtre sur categorie = 'Sport'. Le filtre est écrit après la jointure : à quel endroit apparaît-il dans le plan optimisé ? Comment s'appelle cette optimisation ?

*Indice : commandes.join(produits, "id_produit").filter(F.col("categorie") == "Sport").explain("formatted")*

In [22]:
q = commandes.join(produits, "id_produit").filter(F.col("categorie") == "Sport")
q.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (17)
+- Project (16)
   +- BroadcastHashJoin Inner BuildRight (15)
      :- HashAggregate (5)
      :  +- Exchange (4)
      :     +- HashAggregate (3)
      :        +- Filter (2)
      :           +- Scan csv  (1)
      +- BroadcastExchange (14)
         +- Project (13)
            +- Filter (12)
               +- Window (11)
                  +- Sort (10)
                     +- Exchange (9)
                        +- Project (8)
                           +- Filter (7)
                              +- Scan csv  (6)


(1) Scan csv 
Output [6]: [id_commande#4, id_client#5, id_produit#6, quantite#7, date_commande#8, statut#9]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/data_tp2/commandes.csv]
PushedFilters: [IsNotNull(quantite), IsNotNull(date_commande), IsNotNull(id_client)]
ReadSchema: struct<id_commande:string,id_client:string,id_produit:string,quantite:string,date_commande:string,statut:string>

(2) Filter
Input [6]: [id_comm

**Réponse :** Le filtre est poussé avant la jointure, directement sur le scan de produits (Filter sous Scan, avant BroadcastHashJoin). Il s'agit du « predicate pushdown », une règle de l'optimisation à base de règles de Catalyst.

### Question 8.2

Comme produits est une petite table, Spark a probablement déjà choisi un BroadcastHashJoin en 8.1. Désactivez la diffusion automatique (spark.sql.autoBroadcastJoinThreshold = -1) et observez la stratégie choisie. Puis forcez le broadcast de produits avec F.broadcast(produits) et comparez. Enfin, restaurez la configuration d'origine. Pourquoi le broadcast est-il efficace ici ?

*Indice : spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1") ; spark.conf.unset(...) pour restaurer.*

In [23]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
commandes.join(produits, "id_produit").explain()                 # SortMergeJoin

commandes.join(F.broadcast(produits), "id_produit").explain()    # BroadcastHashJoin

spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")   # retour à la valeur par défaut

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id_produit#277, id_commande#275, id_client#276, quantite#278, date_commande#279, statut#280, nom#1, categorie#2, prix#332]
   +- SortMergeJoin [id_produit#277], [id_produit#330], Inner
      :- Sort [id_produit#277 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(id_produit#277, 8), ENSURE_REQUIREMENTS, [plan_id=7112]
      :     +- HashAggregate(keys=[id_produit#6, date_commande#8, id_commande#4, quantite#7, id_client#5, statut#9], functions=[])
      :        +- Exchange hashpartitioning(id_produit#6, date_commande#8, id_commande#4, quantite#7, id_client#5, statut#9, 8), ENSURE_REQUIREMENTS, [plan_id=7103]
      :           +- HashAggregate(keys=[id_produit#6, date_commande#8, id_commande#4, quantite#7, id_client#5, statut#9], functions=[])
      :              +- Filter (((((((isnotnull(quantite#7) AND isnotnull(date_commande#8)) AND isnotnull(id_client#5)) AND isnotnull(try_cast(quantite#7 as int)))

**Réponse :** Sans diffusion automatique, Spark utilise un SortMergeJoin : les deux tables sont redistribuées (shuffle) puis triées. Avec F.broadcast, la petite table est copiée sur tous les exécuteurs et chaque partition de la grande table est jointe localement, sans shuffle de la grande table sur le réseau.

### Question 8.3

Mesurez l'effet du cache : chronométrez 3 fois le calcul de ventes.groupBy("ville").agg(F.sum("ca")).collect() sans cache, puis après ventes.cache() (et une première action pour remplir le cache). Terminez par unpersist().

*Indice : Utilisez time.time(). Le gain est faible sur de petites données mais le principe reste le même ; consultez l'onglet Storage de l'interface Spark (http://localhost:4040).*

In [35]:
import time

def chrono(n=3):
    durees = []
    for _ in range(n):
        t = time.time()
        ventes.groupBy("ville").agg(F.sum("ca")).collect()
        durees.append(round(time.time() - t, 3))
    return durees

print("Sans cache :", chrono())
ventes.cache()
ventes.count()                 # action qui remplit le cache
print("Avec cache :", chrono())
ventes.unpersist()

Sans cache : [0.685, 0.318, 0.324]
Avec cache : [0.137, 0.104, 0.109]


DataFrame[id_commande: int, date_commande: date, nom_client: string, ville: string, produit: string, categorie: string, quantite: int, prix: double, ca: double]

### Question 8.4

Comparez le plan d'une même agrégation exprimée en DataFrame et en SQL (voir TP1, question 5.4). Que concluez-vous sur le choix de l'API ?

**Réponse :** Les deux écritures donnent le même plan physique car elles passent par Catalyst : on choisit selon la lisibilité et les compétences de l'équipe, pas selon la performance.

## Partie 9 – Persistance et partitionnement

Les résultats sont désormais enregistrés dans des formats adaptés : Parquet pour les données volumineuses réutilisées, CSV pour un export lisible.

### Question 9.1

Ajoutez à ventes les colonnes annee et mois (entiers) et enregistrez-le en Parquet dans out_tp2/ventes partitionné par mois, en mode overwrite. Affichez l'arborescence.

*Indice : F.year("date_commande"), F.month("date_commande") ; write.partitionBy("mois")*

In [25]:
ventes_p = (ventes.withColumn("annee", F.year("date_commande"))
                  .withColumn("mois", F.month("date_commande")))
ventes_p.write.mode("overwrite").partitionBy("mois").parquet("out_tp2/ventes")
print(sorted(d for d in os.listdir("out_tp2/ventes") if d.startswith("mois=")))

['mois=1', 'mois=10', 'mois=11', 'mois=12', 'mois=2', 'mois=3', 'mois=4', 'mois=5', 'mois=6', 'mois=7', 'mois=8', 'mois=9']


### Question 9.2

Relisez le Parquet, filtrez sur mois = 3 et affichez le plan. Repérez PartitionFilters et PushedFilters. Comparez avec un filtre sur la colonne ville (non partitionnée). Quelle différence observez-vous ?

In [26]:
lu = spark.read.parquet("out_tp2/ventes")
lu.filter("mois = 3").explain()
lu.filter("ville = 'Rabat'").explain()

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [id_commande#2098,date_commande#2099,nom_client#2100,ville#2101,produit#2102,categorie#2103,quantite#2104,prix#2105,ca#2106,annee#2107,mois#2108] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/out_tp2/ventes], PartitionFilters: [isnotnull(mois#2108), (mois#2108 = 3)], PushedFilters: [], ReadSchema: struct<id_commande:int,date_commande:date,nom_client:string,ville:string,produit:string,categorie...


== Physical Plan ==
*(1) Filter (isnotnull(ville#2101) AND (ville#2101 = Rabat))
+- *(1) ColumnarToRow
   +- FileScan parquet [id_commande#2098,date_commande#2099,nom_client#2100,ville#2101,produit#2102,categorie#2103,quantite#2104,prix#2105,ca#2106,annee#2107,mois#2108] Batched: true, DataFilters: [isnotnull(ville#2101), (ville#2101 = Rabat)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/out_tp2/ventes], PartitionFilters: [], PushedFilters: [IsNotNu

**Réponse :** Le filtre sur mois apparaît dans PartitionFilters : seuls les dossiers mois=3 sont lus (élagage de partitions). Le filtre sur ville apparaît dans PushedFilters : il est poussé dans le lecteur Parquet qui exploite les statistiques de colonnes, mais tous les fichiers restent parcourus.

### Question 9.3

Produisez le rapport final : un fichier CSV unique (avec en-tête) contenant le CA par catégorie et par mois. Utilisez coalesce(1) pour obtenir un seul fichier, puis affichez son contenu avec Python.

*Indice : df.coalesce(1).write.mode("overwrite").option("header", True).csv("out_tp2/rapport_ca")*
glob.glob(...) : Cette fonction retourne une liste contenant tous les chemins de fichiers qui correspondent à ce motif.

In [27]:
rapport = (ventes_p.groupBy("categorie", "annee", "mois")
                   .agg(F.round(F.sum("ca"), 2).alias("ca"))
                   .orderBy("categorie", "mois"))
rapport.coalesce(1).write.mode("overwrite").option("header", True).csv("out_tp2/rapport_ca")

import glob
fichier = glob.glob("out_tp2/rapport_ca/part-*.csv")[0]
print(open(fichier, encoding="utf-8").read()[:400])

categorie,annee,mois,ca
Informatique,2025,1,35488.0
Informatique,2025,2,26147.0
Informatique,2025,3,29241.0
Informatique,2025,4,27263.0
Informatique,2025,5,30193.0
Informatique,2025,6,24902.0
Informatique,2025,7,31906.0
Informatique,2025,8,42962.0
Informatique,2025,9,25664.0
Informatique,2025,10,30399.0
Informatique,2025,11,25374.0
Informatique,2025,12,24772.0
Maison,2025,1,26724.0
Maison,2025,2,2


### Question 9.4

Pourquoi faut-il utiliser coalesce(1) avec précaution sur de gros volumes ?

**Réponse :** coalesce(1) force toutes les données sur une seule partition, donc un seul exécuteur écrit : on perd le parallélisme et on risque un manque de mémoire. À réserver aux petits résultats finaux.